In [1]:
import json
import re
import random
import matplotlib.pyplot as plt
import lightgbm as lgb
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import pyarrow as pa
from collections import defaultdict
from email_prediction import (
    build_feature_matrix,
    split_clean_ids,
    feature_matrix_batch_loader,
)
from db import TableName, read_table
from sqlalchemy import select, func
from db.models import _TABLE_LOOKUP, get_engine, TableName
from sklearn.metrics import precision_score, recall_score, f1_score
from livelossplot import PlotLosses
from db.db import init_db

# Continued

Last notebook was causing problems, think its too big. Will retry in here.

In [2]:
# Load candidate templates, firm lookups and clean data
candidate_templates = read_table(TableName.CANDIDATE_TEMPLATES)
firm_template_map = read_table(TableName.FIRM_TEMPLATE_MAP)
lp_df = read_table(TableName.LP_CLEAN)

Read CANDIDATE_TEMPLATES table from database!
Read FIRM_TEMPLATE_MAP table from database!
Read LP_CLEAN table from database!


In [3]:
# Define complex flags
complex_flags = [
    "has_multiple_first_names",
    "has_middle_name",
    "has_multiple_middle_names",
    "has_multiple_last_names",
    "has_nfkd_normalized",
    "has_german_char",
]

# Create a single column flag
lp_df["is_name_complex"] = lp_df[complex_flags].any(axis=1)

# Split into two datasets
complex_name_df = lp_df[lp_df["is_name_complex"]].copy()
standard_name_df = lp_df[~lp_df["is_name_complex"]].copy()

print(f"Complex names: {len(complex_name_df)}")
print(f"Standard names: {len(standard_name_df)}")
print(lp_df["is_name_complex"].value_counts(normalize=True))

# JSON safe string form of token_seq
complex_name_df["token_seq_str"] = complex_name_df["token_seq"].apply(json.dumps)
standard_name_df["token_seq_str"] = standard_name_df["token_seq"].apply(json.dumps)
candidate_templates["token_seq_str"] = candidate_templates["template"].apply(json.dumps)

# Unique sequences seen in each subset
complex_token_set = set(complex_name_df["token_seq_str"])
standard_token_set = set(standard_name_df["token_seq_str"])

# Filter candidate_templates
complex_templates = candidate_templates[
    candidate_templates["template"].isin(complex_token_set)
].copy()
standard_templates = candidate_templates[
    candidate_templates["template"].isin(standard_token_set)
].copy()

print(f"Complex templates: {len(complex_templates)}")
print(f"Standard templates: {len(standard_templates)}")


def build_firm_profile(
    firm_id: str,
    lp_clean_df: pd.DataFrame,
    firm_template_map_df: pd.DataFrame,
    candidate_templates_df: pd.DataFrame,
) -> dict:
    firm_investors = (
        lp_clean_df[lp_clean_df["firm"] == firm_id].dropna(subset=["token_seq"]).copy()
    )
    firm_stats = firm_template_map_df[firm_template_map_df["firm"] == firm_id]
    if firm_investors.empty or firm_stats.empty:
        return None

    name_flags = [
        "has_german_char",
        "has_nfkd_normalized",
        "has_nickname",
        "has_multiple_first_names",
        "has_middle_name",
        "has_multiple_middle_names",
        "has_multiple_last_names",
    ]

    # Firm-wide flags
    infra_flag = firm_investors["is_shared_infra"].iloc[0]
    multi_domain_flag = firm_investors["firm_is_multi_domain"].iloc[0]

    # Resolve template IDs and pull their data
    template_ids_raw = firm_stats["template_ids"].values[0]
    template_ids = (
        json.loads(template_ids_raw)
        if isinstance(template_ids_raw, str)
        else template_ids_raw
    )
    templates = candidate_templates_df[
        candidate_templates_df["template_id"].isin(template_ids)
    ].copy()
    if templates.empty:
        return None

    # Canonicalize as tuples for lookup
    firm_investors["template_tuple"] = firm_investors["token_seq"].apply(tuple)
    templates["template_tuple"] = templates["template"].apply(
        lambda x: tuple(x) if isinstance(x, list) else tuple(json.loads(x))
    )

    # Count template usage and flag distributions
    template_usage = firm_investors["template_tuple"].value_counts().to_dict()
    template_flag_stats = {
        tmpl: firm_investors[firm_investors["template_tuple"] == tmpl][name_flags]
        .mean(numeric_only=True)
        .to_dict()
        for tmpl in templates["template_tuple"]
    }

    # Find template sample rate
    total_usage = sum(template_usage.get(t, 0) for t in templates["template_tuple"])
    templates["sampling_weight"] = templates["template_tuple"].apply(
        lambda t: template_usage.get(t, 0) / total_usage if total_usage > 0 else 0.0
    )

    # Compute mean for probalistic sampling
    templates["flag_probs"] = templates["template_tuple"].apply(
        lambda t: template_flag_stats.get(t, {k: 0.0 for k in name_flags})
    )

    template_records = templates[["template", "sampling_weight", "flag_probs"]].to_dict(
        orient="records"
    )

    return {
        "firm": firm_id,
        "is_shared_infra": infra_flag,
        "firm_is_multi_domain": multi_domain_flag,
        "templates": template_records,
    }


def generate_synthetic_investors(
    profile: dict, n_padding: int, starting_id: int = 1000000
) -> tuple[pd.DataFrame, int]:
    curr_id = starting_id
    # Unpack profile
    firm = profile["firm"]
    is_shared_infra = profile["is_shared_infra"]
    firm_is_multi_domain = profile["firm_is_multi_domain"]
    templates = profile["templates"]

    if not templates:
        raise ValueError(f"No templates available for firm {firm}")

    rows = []

    # Generate rows
    for i in range(n_padding):
        curr_id += i

        # Sample a template and use its flag probabilities
        chosen_template = random.choices(
            templates, weights=[t["sampling_weight"] for t in templates], k=1
        )[0]

        flag_probs = chosen_template["flag_probs"]
        name_flags = list(flag_probs.keys())

        # Construct row
        row = {
            "id": curr_id,
            "firm": firm,
            "investor": f"synthetic_investor_{firm}_{i}",
            "is_shared_infra": is_shared_infra,
            "firm_is_multi_domain": firm_is_multi_domain,
            "token_seq": json.loads(chosen_template["template"]),
        }

        # Sample name flags
        for flag in name_flags:
            flag_prob = flag_probs.get(flag)
            if flag_prob is None:
                print(f"{flag} missing in {firm} profile!")
            row[flag] = bool(np.random.binomial(1, flag_probs.get(flag, 0.0)))

        rows.append(row)

    return pd.DataFrame(rows), curr_id


def padd_with_synthetic(df, firm_profiles, n_pad=10):
    # Take a fresh copy
    augmented = df.copy()

    # Generate synethic rows
    all_synthetic_rows = []
    last_id = 1000000
    for profile in firm_profiles:
        # Will start with padding by 10
        if n_pad > 0:
            synthetic_rows, last_id = generate_synthetic_investors(
                profile, n_padding=n_pad, starting_id=last_id
            )
            all_synthetic_rows.append(synthetic_rows)

    # Combine all synthetic rows into one DataFrame
    synthetic_df = pd.concat(all_synthetic_rows, ignore_index=True)

    # Merge back into LP
    augmented = pd.concat([augmented, synthetic_df], ignore_index=True)

    return augmented

Complex names: 4000
Standard names: 69916
is_name_complex
False    0.945885
True     0.054115
Name: proportion, dtype: float64
Complex templates: 355
Standard templates: 79


In [ ]:
# More aggressive padding
low_investor_firms = firm_template_map[firm_template_map["num_investors"] < 10][
    "firm"
].tolist()

# Build firm profiles for each data set
firm_profiles_std = [
    build_firm_profile(firm, standard_name_df, firm_template_map, standard_templates)
    for firm in low_investor_firms
]
firm_profiles_com = [
    build_firm_profile(firm, complex_name_df, firm_template_map, complex_templates)
    for firm in low_investor_firms
]
firm_profiles_std = [x for x in firm_profiles_std if x is not None]
firm_profiles_com = [x for x in firm_profiles_com if x is not None]

# Padd both with synthetic
std_augmented = padd_with_synthetic(standard_name_df, firm_profiles_std, n_pad=20)
complex_augmented = padd_with_synthetic(complex_name_df, firm_profiles_com, n_pad=100)

# Get splits for both
_, val_std, test_std = split_clean_ids(
    standard_name_df, test_ratio=0.1, validation_ratio=0.2
)
pd.DataFrame({"validation_ids": val_std}).to_csv("validation_std_ids.csv")
all_ids = std_augmented["id"].to_list()
train_std = list(set(all_ids) - set(val_std) - set(test_std))
pd.DataFrame({"train_ids": train_std}).to_csv("train_std_ids.csv")
_, val_comp, test_comp = split_clean_ids(
    complex_name_df, test_ratio=0.1, validation_ratio=0.2
)
pd.DataFrame({"validation_ids": val_comp}).to_csv("validation_com_ids.csv")
all_ids = complex_augmented["id"].to_list()
train_comp = list(set(all_ids) - set(val_comp) - set(test_comp))
pd.DataFrame({"train_ids": train_comp}).to_csv("train_com_ids.csv")

Save to separate feature matrices.

In [5]:
# Isolate training and validation set
training_and_validation_set_std = std_augmented[
    std_augmented["id"].isin(list(train_std + val_std))
]
training_and_validation_set_comp = complex_augmented[
    complex_augmented["id"].isin(list(train_comp + val_comp))
]
# Save to DB
init_db()
build_feature_matrix(
    training_and_validation_set_std,
    standard_templates,
    firm_template_map,
    table=TableName.FEATURE_MATRIX,
)
build_feature_matrix(
    training_and_validation_set_comp,
    complex_templates,
    firm_template_map,
    table=TableName.FEATURE_MATRIX_COMPLEX,
)

Tables created successfully.
0 features built out of 327685
79 upserted, 0 inserted into FEATURE_MATRIX
79 upserted, 0 inserted into FEATURE_MATRIX
79 upserted, 0 inserted into FEATURE_MATRIX
79 upserted, 0 inserted into FEATURE_MATRIX
79 upserted, 0 inserted into FEATURE_MATRIX
79 upserted, 0 inserted into FEATURE_MATRIX
79 upserted, 0 inserted into FEATURE_MATRIX
79 upserted, 0 inserted into FEATURE_MATRIX
79 upserted, 0 inserted into FEATURE_MATRIX
79 upserted, 0 inserted into FEATURE_MATRIX
79 upserted, 0 inserted into FEATURE_MATRIX
79 upserted, 0 inserted into FEATURE_MATRIX
79 upserted, 0 inserted into FEATURE_MATRIX
79 upserted, 0 inserted into FEATURE_MATRIX
79 upserted, 0 inserted into FEATURE_MATRIX
79 upserted, 0 inserted into FEATURE_MATRIX
79 upserted, 0 inserted into FEATURE_MATRIX
79 upserted, 0 inserted into FEATURE_MATRIX
79 upserted, 0 inserted into FEATURE_MATRIX
79 upserted, 0 inserted into FEATURE_MATRIX
79 upserted, 0 inserted into FEATURE_MATRIX
79 upserted, 0 i